In [39]:
import pandas as pd
import re
import numpy as np
from datetime import datetime

In [40]:
def remove_message_timestamp(message):
    regex = r"'time': \d+, "
    new_message = re.sub(regex, '', str(message))
    regex = r"'dpId': \d+, "
    new_message = re.sub(regex, '', str(new_message))
    return eval(new_message)

def remove_invalid_codes(message):
    valid_codes = ['switch_led', 'switch_1', 'presence_state', 'doorcontact_state']
    result = []
    for state in message:
        if state['code'] in valid_codes:
            result.append(state)
    if len(result) > 0:
        return result
    else:
        return np.nan

In [41]:
def trata_dados_reais(caminho:str):
    message_list = []
    hour_list = []
    with open(caminho, "r") as json:
        data = json.read()
        for line in data.split('\n')[1:-1]:
            hour_list.append(eval(line)['hora'])
            dict_msg = eval(eval(line)['mensagem'].replace('true', 'True').replace('false', 'False'))['bizData']
            message_list.append(dict_msg)
        json.close()
    
    df = pd.DataFrame(message_list).rename(columns={'properties': 'message'})
    df['timeStamp'] = hour_list
    df['message'] = df['message'].astype(str)
    
    dicionario = {
        'ebfe6c248a7dfe6910qdcb': 'Plug_fan',
        'eb061b979815289561tyqf': 'Sensor_presence',
        'eb31770a1d7812125degzr': 'Light_bulb',
        'eb176a71685a57c19arlbp': 'abertura',
        'ebcc9b86347718a3808ezt': 'Plug_pc',
        'ebf1d890916d1a73b4vtnv': 'temp_humidade'
    }
    
    df['devId'] = df['devId'].apply(lambda x: dicionario[x])
    
    df['message'] = df['message'].apply(remove_message_timestamp)
    df['message'] = df['message'].apply(remove_invalid_codes)
    df['timeStamp'] = df['timeStamp'].apply(lambda string: datetime.strptime(string, '%Y-%m-%dT%H:%M:%S.%f'))

    df = df[df['message'].notna()].reset_index(drop=True)
    df["day_week"] = df["timeStamp"].dt.day_name()

    unique_devices = ['Sensor_presence', 'Plug_fan', 'Light_bulb', 'Plug_pc', 'abertura', 'temp_humidade']

    for device in unique_devices:
        result = (df[df['devId'] == device]['message'].shift(1) == df[df['devId'] == device]['message']).sum()
        print(f'{device}: {result}')

    remove_index = []
    for device in unique_devices:
        result = df[df['devId'] == device][(df[df['devId'] == device]['message'].shift(1) == df[df['devId'] == device]['message']).values].index.tolist()
        if len(result) > 0:
            remove_index.extend(result)
    df = df.drop(remove_index)
    return df

In [42]:
df_teste = trata_dados_reais('./dados_reais_14_dias.json')

Sensor_presence: 1044
Plug_fan: 0
Light_bulb: 5
Plug_pc: 0
abertura: 5
temp_humidade: 0


In [43]:
df_teste

,devId,dataId,productId,message,timeStamp,day_week
0,Plug_fan,00061ED27C47CD3BD6F7A82A668E0008,2vdttet77fzapqdf,"[{'code': 'switch_1', 'value': True}]",2024-08-03 23:40:37.634547,Saturday
1,Plug_fan,00061ED27E0B4C12D6F7A82A668E000B,2vdttet77fzapqdf,"[{'code': 'switch_1', 'value': False}]",2024-08-03 23:41:07.239471,Saturday
2,Sensor_presence,00061ED280093CB4DE9AA33966E30116,k2h8vkj98fhvnpiv,"[{'code': 'presence_state', 'value': 'none'}]",2024-08-03 23:41:40.799322,Saturday
3,Sensor_presence,00061ED282825DC7DE9AA33966E30118,k2h8vkj98fhvnpiv,"[{'code': 'presence_state', 'value': 'presence'}]",2024-08-03 23:42:22.193604,Saturday
5,Plug_fan,00061ED2A35E172AD6F7A82A668E0013,2vdttet77fzapqdf,"[{'code': 'switch_1', 'value': True}]",2024-08-03 23:51:33.407493,Saturday
...,...,...,...,...,...,...
4692,Sensor_presence,00062007C04A3C4DFBBE514366730595,k2h8vkj98fhvnpiv,"[{'code': 'presence_state', 'value': 'none'}]",2024-08-19 08:38:43.540502,Monday
4693,Sensor_presence,00062007C0BB2964FBBE514366730597,k2h8vkj98fhvnpiv,"[{'code': 'presence_state', 'value': 'presence'}]",2024-08-19 08:38:50.910507,Monday
4694,Sensor_presence,00062007C25605A8FBBE5143667305B3,k2h8vkj98fhvnpiv,"[{'code': 'presence_state', 'value': 'none'}]",2024-08-19 08:39:17.816938,Monday
4695,Sensor_presence,00062007D44E6109FBBE5143667305B7,k2h8vkj98fhvnpiv,"[{'code': 'presence_state', 'value': 'presence'}]",2024-08-19 08:44:19.265872,Monday


In [49]:
df_teste[df_teste['devId'] == 'Sensor_presence']

,devId,dataId,productId,message,timeStamp,day_week
2,Sensor_presence,00061ED280093CB4DE9AA33966E30116,k2h8vkj98fhvnpiv,"[{'code': 'presence_state', 'value': 'none'}]",2024-08-03 23:41:40.799322,Saturday
3,Sensor_presence,00061ED282825DC7DE9AA33966E30118,k2h8vkj98fhvnpiv,"[{'code': 'presence_state', 'value': 'presence'}]",2024-08-03 23:42:22.193604,Saturday
11,Sensor_presence,00061ED2AB42B76E0CCC52396651000E,k2h8vkj98fhvnpiv,"[{'code': 'presence_state', 'value': 'none'}]",2024-08-03 23:53:45.952854,Saturday
12,Sensor_presence,00061ED2AC025A420CCC523966510011,k2h8vkj98fhvnpiv,"[{'code': 'presence_state', 'value': 'presence'}]",2024-08-03 23:53:58.428507,Saturday
14,Sensor_presence,00061ED2CC2804090A38762A66BB0000,k2h8vkj98fhvnpiv,"[{'code': 'presence_state', 'value': 'none'}]",2024-08-04 00:02:57.711475,Sunday
...,...,...,...,...,...,...
4686,Sensor_presence,000620047DDC63C498A363436667000A,k2h8vkj98fhvnpiv,"[{'code': 'presence_state', 'value': 'presence'}]",2024-08-19 04:45:24.183379,Monday
4692,Sensor_presence,00062007C04A3C4DFBBE514366730595,k2h8vkj98fhvnpiv,"[{'code': 'presence_state', 'value': 'none'}]",2024-08-19 08:38:43.540502,Monday
4693,Sensor_presence,00062007C0BB2964FBBE514366730597,k2h8vkj98fhvnpiv,"[{'code': 'presence_state', 'value': 'presence'}]",2024-08-19 08:38:50.910507,Monday
4694,Sensor_presence,00062007C25605A8FBBE5143667305B3,k2h8vkj98fhvnpiv,"[{'code': 'presence_state', 'value': 'none'}]",2024-08-19 08:39:17.816938,Monday
